In [7]:
from pathlib import Path
import sys

def find_project_root():
    here = Path.cwd()
    for p in [here, *here.parents]:
        if (p / "utils").exists() and (p / "models").exists():
            return p
    for p in [here, *here.parents]:
        if any((p / m).exists() for m in (".git", "requirements.txt", "pyproject.toml")):
            return p
    return here

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import sys
import os
import torch
import glob
import importlib

# 1. SETUP PATHS
PROJECT_ROOT = os.path.abspath(".")
TIMEPFN_ROOT = os.path.join(PROJECT_ROOT, "TimePFN")

if TIMEPFN_ROOT not in sys.path:
    sys.path.insert(0, TIMEPFN_ROOT)

# Clean imports to ensure safety
keys_to_remove = [k for k in sys.modules if k.startswith('utils')]
for k in keys_to_remove:
    del sys.modules[k]

try:
    from model.TimePFN import Model as TimePFN
    print(f" TimePFN imported")
except ImportError:
    from model.TimePFN import Model as TimePFN

# Import Adapter
from utils.timepfn_adapter import TimePFNAdapter
from utils.data_loadingV4_file import apply_threshold_v1_logic

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# 2. CONFIG (Must match the Fine-Tuning Script)
class TimePFNConfig:
    def __init__(self, seq_len=96, pred_len=96, n_vars=5):
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.enc_in = n_vars  # Matches --enc_in 5
        self.dec_in = n_vars  # Matches --dec_in 5
        self.c_out = n_vars   # Matches --c_out 5
        
        # Arch params from script
        self.d_model = 1024
        self.d_ff = 512
        self.n_heads = 8
        self.e_layers = 8
        self.d_layers = 1
        self.dropout = 0.1
        self.patch_size = 16
        self.embed_dim = 256
        
        # Flags
        self.use_norm = True
        self.activation = 'gelu'
        self.output_attention = False
        self.embed = 'timeF'
        self.freq = 'h'
        self.factor = 1
        self.use_gpu = True if device == 'cuda' else False
        
        # Safe Defaults
        self.class_strategy = 'projection'
        self.num_class = 1
        self.fc_dropout = 0.1
        self.head_dropout = 0.1
        self.down_sampling_layers = 0
        self.down_sampling_method = 'avg'
        self.down_sampling_window = 1

# Initialize
FEATURES = ['handwheelAngle', 'vxCG', 'yawRate', 'ayCG'] 
N_VARS = len(FEATURES) + 1 
CONTEXT_LEN = 96

configs = TimePFNConfig(seq_len=CONTEXT_LEN, pred_len=CONTEXT_LEN, n_vars=N_VARS)
model = TimePFN(configs).to(device)

# 3. FIND & LOAD FINE-TUNED WEIGHTS
# Search for the folder created by your bash script
ft_checkpoints = glob.glob(os.path.join(TIMEPFN_ROOT, "checkpoints/sensing_drift_ft_*/checkpoint.pth"))

if ft_checkpoints:
    # Get the most recent one
    WEIGHTS_PATH = max(ft_checkpoints, key=os.path.getmtime)
    print(f"\n Loading FINE-TUNED weights from:\n   {WEIGHTS_PATH}\n")
else:
    print("\n Fine-tuned weights not found. Using Base Model.\n")
    WEIGHTS_PATH = os.path.join(TIMEPFN_ROOT, "checkpoints/TimePFN/checkpoint.pth")

model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device), strict=False)
model.eval()

# 4. LOAD CONTEXT
# We use a balanced context to help the model recall drift physics
adapter = TimePFNAdapter(train_dir="datasets_challenge/train", history_len=CONTEXT_LEN, feature_cols=FEATURES)
ctx_x, ctx_y = adapter.get_balanced_context(n_drift=12, n_normal=52)
ctx_x, ctx_y = ctx_x.to(device), ctx_y.to(device)

print(" Model Ready for Inference")

✅ TimePFN imported
Using device: cuda

🚀 Loading FINE-TUNED weights from:
   /home/mdesouky/workspace/mait_g2_drift_challenge/TimePFN/checkpoints/sensing_drift_ft_TimePFN_custom_M_ft96_sl48_ll96_pl1024_dm8_nh8_el1_dl512_df1_fctimeF_ebTrue_dtExp_projection_0_embDim256_dataAmount-1_maxChannel161_subset-1/checkpoint.pth

 Loading and Filtering Training Data (vxCG > 3.0)...
 Clean Physics Pool: 137553 rows
 Fitting StandardScaler...
 Balancing Pool Ready: 12048 Drift vs 125408 Normal
✅ Model Ready for Inference


In [10]:
import glob
import pandas as pd
import numpy as np
from tqdm import tqdm
from scipy.signal import savgol_filter

# CONFIG
OLD_SEQ_LEN = 20  # Matches Kaggle row count
TIMEPFN_CTX = 96  # Model requirement
THRESHOLD_VAL = 3.0

test_files = sorted(glob.glob("datasets_challenge/test/test_part_*.csv"))
scaler = adapter.get_scaler()

final_predictions = []
final_ids = []
global_id_counter = 0

print(f"Processing {len(test_files)} files...")

for f in test_files:
    df = pd.read_csv(f)
    
    # 1. APPLY EXACT FILTERING LOGIC
    valid_groups = apply_threshold_v1_logic(
        df, 
        threshold_column="vxCG", 
        threshold_value=THRESHOLD_VAL, 
        seq_length=OLD_SEQ_LEN
    )
    
    if not valid_groups:
        continue

    # 2. PROCESS GROUPS
    for group in valid_groups:
        # Scale Group
        group_scaled = group.copy()
        group_scaled['sideSlip'] = 0.0
        cols_to_scale = FEATURES + ['sideSlip']
        group_scaled[cols_to_scale] = scaler.transform(group_scaled[cols_to_scale])
        scaled_feats = group_scaled[FEATURES].values
        
        # Track IDs
        num_preds_in_group = len(group) - OLD_SEQ_LEN
        current_ids = np.arange(global_id_counter, global_id_counter + num_preds_in_group)
        final_ids.extend(current_ids)
        global_id_counter += num_preds_in_group
        
        batch_preds = []
        
        for i in range(num_preds_in_group):
            # Target Index relative to group
            target_idx_in_group = i + OLD_SEQ_LEN - 1
            
            # Prepare Window
            w_end = target_idx_in_group + 1
            w_start = w_end - TIMEPFN_CTX
            
            if w_start < 0:
                valid_data = scaled_feats[0 : w_end]
                pad_len = TIMEPFN_CTX - len(valid_data)
                padding = np.zeros((pad_len, len(FEATURES)))
                window = np.vstack([padding, valid_data])
            else:
                window = scaled_feats[w_start : w_end]
            
            curr_x = torch.tensor(window).float().unsqueeze(0).to(device)
            # Dummy Y
            dummy_y = torch.zeros((1, TIMEPFN_CTX, 1)).to(device)
            
            with torch.no_grad():
                output = model(ctx_x, ctx_y, curr_x, dummy_y)
                val = output[0, -1, -1].item()
                batch_preds.append(val)
                
        final_predictions.extend(batch_preds)

# 3. INVERSE TRANSFORM
n_samples = len(final_predictions)
dummy_feats = np.zeros((n_samples, len(FEATURES)))
pred_col = np.array(final_predictions).reshape(-1, 1)
matrix_to_inverse = np.hstack([dummy_feats, pred_col])
real_degrees_pred = scaler.inverse_transform(matrix_to_inverse)[:, -1]

# ==========================================
# 4. POST-PROCESSING (THE FIX FOR MAE)
# ==========================================

print(" Applying Post-Processing...")

# A. Gating (Zero-Clamping)
# Force small jitters (< 0.5 degrees) to 0.0.
# This fixes the 'Many Shot' error on straight roads.
CLAMP_THRESHOLD = 0.5
real_degrees_pred[np.abs(real_degrees_pred) < CLAMP_THRESHOLD] = 0.0

# B. Smoothing (Savitzky-Golay)
# Removes high-freq noise
try:
    real_degrees_pred = savgol_filter(real_degrees_pred, window_length=11, polyorder=2)
except ValueError:
    pass

# C. Clamp Again (Cleanup after smoothing)
real_degrees_pred[np.abs(real_degrees_pred) < CLAMP_THRESHOLD] = 0.0

# 5. SAVE
submission = pd.DataFrame({
    "id": final_ids,
    "predicted_sideslip": real_degrees_pred
})
submission.to_csv("submission_timepfn_finetuned.csv", index=False)

print(f"Saved submission_timepfn_finetuned.csv")
print(f" Final Count: {len(submission)}")
if len(submission) == 49328:
    print(" SUCCESS! Count matches requirement.")

Processing 11 files...
🔧 Applying Post-Processing...
✅ Saved submission_timepfn_finetuned.csv
📊 Final Count: 49328
🎉 SUCCESS! Count matches requirement.
